In [ ]:
# Importing libraries and config

from pathlib import Path

from IPython.display import display
from PIL import Image, ImageOps

from object_detection.config.loader import load_config
from object_detection.detection.inference import Detector
from object_detection.identification.identify import identify
from object_detection.identification.index import create_index
from object_detection.utils.visualization import draw_identifications

cfg = load_config()
identification = cfg.identification

detector = Detector(cfg.inference.weights, cfg.inference.conf)
index = create_index(identification)

print(f"detector: {cfg.inference.weights.name} at conf {cfg.inference.conf}")
print(f"index: {identification.collection} ({identification.model})")

In [ ]:
# Running the whole pipeline on one photo

def show(image_path, max_size=900):
    with Image.open(image_path) as opened:
        image = ImageOps.exif_transpose(opened).convert("RGB")
        detections = detector.predict(image)
        results = identify(
            image, detections, index,
            padding=identification.crop_padding,
            min_size=identification.min_crop_size,
            unknown_threshold=identification.unknown_threshold,
        )
        annotated = draw_identifications(image, results)

    print(f"{len(results)} products detected")
    for result in results:
        name = result.object_name or "unknown"
        score = f"{result.match_score:.2f}" if result.match_score is not None else "-"
        print(f"    conf {result.detection.confidence:.2f}  {name:<16} match {score}")

    annotated.thumbnail((max_size, max_size))
    display(annotated)

In [ ]:
# Running the demo

show(r"C:\Users\Nitro\Documents\GitHub\everyday-object-detection\test_images\testimg1.jpg")

In [ ]:
# Releasing the index so Qdrant local mode doesn't lock the folder

index.close()